# Notebook limpio de manin, aquí probaremos nuevas estrategias con orderflow

In [1]:
import numpy as np
import pandas as pd
import sys
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sys.path.append(os.path.abspath("../src"))
from data_cleaner import DataCleanerConfig, DataCleaner, preprocess_data


## Importamos los datos 

In [2]:
cfg_5m = DataCleanerConfig(
    source="alpaca",
    symbol=["QQQ", "TLT", "VXX", "BNDX"],
    interval="5m",
    start_date="2022-01-01",
    end_date="2026-02-02",
    
)

cleaner = DataCleaner(cfg_5m)
df_raw = cleaner.cargar_datos()

df = preprocess_data(df_raw, "qqq")

df.head()


c:\Users\koki2\Desktop\TRADING_ALG\src\data_cleaner.py:242: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df["close"].pct_change()


,datetime,open_bndx,open_qqq,open_tlt,open_vxx,high_bndx,high_qqq,high_tlt,high_vxx,low_bndx,...,log_return_lag5,vol_rolling,rsi,macd,macd_signal,ema_9,ema_21,ema_50,ema_100,ema_200
0,2022-01-03 14:25:00+00:00,55.06,399.38,146.55,18.2100,55.060,399.41,146.600,18.30,55.06,...,-0.000275,0.000512,21.777778,-0.000707,-0.000518,399.110000,399.110000,399.110000,399.110000,399.110000
1,2022-01-03 14:30:00+00:00,55.03,399.05,146.41,18.2700,55.060,400.68,146.500,18.29,55.01,...,0.000601,0.000921,50.991501,-0.000474,-0.000509,399.372000,399.229091,399.161373,399.135941,399.123035
2,2022-01-03 14:35:00+00:00,55.02,400.44,146.36,18.1200,55.035,401.53,146.595,18.16,55.01,...,-0.000025,0.000983,60.246913,-0.000158,-0.000439,399.709600,399.395537,399.235829,399.174041,399.142308
3,2022-01-03 14:40:00+00:00,55.03,401.07,146.57,18.0701,55.040,401.26,146.800,18.15,55.03,...,0.000326,0.001024,57.547170,-0.000008,-0.000353,399.879680,399.501397,399.287757,399.201486,399.156414
4,2022-01-03 14:45:00+00:00,55.03,400.57,146.53,18.1100,55.050,400.75,146.940,18.56,55.03,...,-0.000050,0.001669,36.489076,-0.000359,-0.000354,399.547244,399.384679,399.245786,399.182001,399.147072


# Definimos el target
*df["ret_fwd"] = df["close"].shift(-H) / df["close"] - 1* -> Crea una nueva columna ret_fwd y calcula el retorno porcentual (ejemplo: si hoy close es 100, en H velas close = 101 -> 1%)

*df.loc[df["ret_fwd"] >  thr, "y"] = 1* -> Crea la etiqueta binaria Y.  y = 1 si el retorno futuro es positivo y 0 si es negativo

# Resumen: 
Calculamos el retorno futuro a H velas (ret_fwd).

Solo etiquetamos como sube (1) si el movimiento supera +thr y como baja (0) si baja más de -thr.

Todo lo que sea “ruido” (entre -thr y +thr) lo descartamos para entrenar con ejemplos más claros.

In [ ]:
H = 24          # horizonte 
thr = 0.0005    # 0.05% = 5 bps 

df = df.copy()
df["ret_fwd"] = df["close"].shift(-H) / df["close"] - 1

df["y"] = np.nan
df.loc[df["ret_fwd"] >  thr, "y"] = 1
df.loc[df["ret_fwd"] < -thr, "y"] = 0

df = df.dropna(subset=["ret_fwd", "y"]).reset_index(drop=True)
df["y"] = df["y"].astype(int)


*df["ret_1"] = df["close"] / df["close"].shift(1) - 1* -> Crea el retorno de 1 vela hacia atrás (ret_1)

Dividimos train y test en 70% train, 30% test

*majority = int(train["y"].mean() >= 0.5)* -> Decide cuál es la clase más común en el train

*test["pred_majority"] = majority* -> Predice siempre esa clase (0 o 1) para todas las filas del test

*test["pred_mom1"] = (test["ret_1"] > 0).astype(int)* -> Si la vela anterior subió (ret_1 > 0) predice 1, si no predice 0.

In [7]:
# Feature ultra simple (solo pasado)
df = df.copy()
df["ret_1"] = df["close"] / df["close"].shift(1) - 1
df = df.dropna(subset=["ret_1"]).reset_index(drop=True)

# Split temporal (sin shuffle)
split = int(len(df) * 0.70)
train = df.iloc[:split].copy()
test  = df.iloc[split:].copy()

# Baseline 1: predice siempre la clase mayoritaria del train
majority = int(train["y"].mean() >= 0.5)
test["pred_majority"] = majority

# Baseline 2: momentum 1 vela (si la última vela subió, predice 1)
test["pred_mom1"] = (test["ret_1"] > 0).astype(int)


In [8]:
acc_majority = (test["pred_majority"] == test["y"]).mean()
acc_mom1 = (test["pred_mom1"] == test["y"]).mean()

print("Train size:", len(train), " Test size:", len(test))
print("Train y mean:", round(train["y"].mean(), 3), " Test y mean:", round(test["y"].mean(), 3))
print("Majority class:", majority)
print("ACC majority:", round(acc_majority, 4))
print("ACC mom1:", round(acc_mom1, 4))

test[["datetime","close","ret_1","ret_fwd","y","pred_majority","pred_mom1"]].head(10)


Train size: 51905  Test size: 22245
Train y mean: 0.535  Test y mean: 0.558
Majority class: 1
ACC majority: 0.5581
ACC mom1: 0.513


,datetime,close,ret_1,ret_fwd,y,pred_majority,pred_mom1
51905,2024-10-31 14:00:00+00:00,487.3500,-0.000533,-0.004330,0,1,0
51906,2024-10-31 14:05:00+00:00,487.3492,-0.000002,-0.003959,0,1,0
51907,2024-10-31 14:10:00+00:00,486.8400,-0.001045,-0.002485,0,1,0
51908,2024-10-31 14:15:00+00:00,488.1300,0.002650,-0.004712,0,1,1
51909,2024-10-31 14:20:00+00:00,488.2200,0.000184,-0.005162,0,1,1
51910,2024-10-31 14:25:00+00:00,486.9700,-0.002560,-0.002587,0,1,0
51911,2024-10-31 14:30:00+00:00,486.7200,-0.000513,-0.002794,0,1,0
51912,2024-10-31 14:35:00+00:00,486.7600,0.000082,-0.002568,0,1,1
51913,2024-10-31 14:40:00+00:00,486.5600,-0.000411,-0.001233,0,1,0
51914,2024-10-31 14:50:00+00:00,485.6700,-0.001829,0.000597,1,1,0
